# LGEO2270 | Management scenarios
**UCLouvain · Physical geography excursion**

Upload the unchanged NB01 Baseline ZIP. Design an intervention, run the annual and 40 mm simulations, and save the Scenario ZIP.

Choose **Programmed** for automatic field selection, or **GIS upload** to design interventions in QGIS or ArcGIS.

[GIS scenario quick guide](https://github.com/kvanoost/Watem/blob/main/LGEO2270_GIS_scenario_guide.md) · [Model methodology](https://github.com/kvanoost/Watem/blob/main/LGEO2270_Model_methodology.md)

In [ ]:
#@title 1 — Upload baseline
from google.colab import files
from pathlib import Path
import os, shutil, json, re, zipfile, tempfile
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from rasterio.features import rasterize
from shapely.ops import unary_union
from shapely.geometry import box
from IPython.display import display
ROOT=Path(tempfile.mkdtemp(prefix='watem_local_'))
import urllib.request, sys
helper_dir=Path('/content/watem_student_helpers'); helper_dir.mkdir(exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/kvanoost/Watem/50839661b3a72d2f7a78b7c3d0a4ba7019908cc7/watem_student_common.py',helper_dir/'watem_student_common.py')
sys.path.insert(0,str(helper_dir))
import watem_student_common as ws

print('Upload the Baseline ZIP from NB01')
uploads=files.upload()
packages={}
for i,(name,data) in enumerate(uploads.items()):
    archive=ROOT/f'upload_{i}.zip'; archive.write_bytes(data)
    folder=ws.safe_extract(archive,ROOT/f'package_{i}')
    packages[name]=folder
baseline_hits=[]
for folder in packages.values():
    try: baseline_hits.append(ws.identify_package(folder,'baseline'))
    except AssertionError: pass
assert len(baseline_hits)==1,'Upload exactly one NB01 Baseline ZIP'
BASELINE,manifest=baseline_hits[0]; SITE_ID=manifest['site']; SITE=ROOT/'site'; SITE.mkdir()
assert ws.baseline_digest(BASELINE)==manifest['baseline_id'],'Baseline inputs do not match manifest'
print('Site:',SITE_ID,'baseline:',manifest['baseline_id'][:12])


## 2. Design your scenario
For GIS upload, edit the two management columns in `management.shp` and optionally draw grass strips in `grass.shp`. ZIP the editable_scenario folder. [Quick guide](https://github.com/kvanoost/Watem/blob/main/LGEO2270_GIS_scenario_guide.md).

In [ ]:
#@title 2 — Design management scenario
SCENARIO_NAME='S01_my_design' #@param {type:'string'}
assert re.fullmatch(r'[A-Za-z0-9_-]{1,60}',SCENARIO_NAME) and SCENARIO_NAME.lower()!='baseline'
DEST=SITE/'scenarios'/SCENARIO_NAME
assert not DEST.exists(),'Use a unique name; never overwrite a scenario'
MODE='Programmed' #@param ['Programmed','GIS upload']
work=Path(tempfile.mkdtemp(prefix='watem_scenario_'))
layers={}
if MODE=='GIS upload':
    print('Upload your edited management.shp and/or grass.shp as a ZIP.')
    uploaded=files.upload(); assert len(uploaded)==1
    archive=work/'original.zip'; archive.write_bytes(next(iter(uploaded.values())))
    ws.safe_extract(archive,work/'design')
    for name in ['C_change','CN_change','grass']:
        hits=list((work/'design').rglob(name+'.shp')); assert len(hits)<=1
        if hits:
            g=gpd.read_file(hits[0]); assert g.crs is not None
            g=g.to_crs(31370); assert g.geometry.is_valid.all() and not g.geometry.is_empty.any()
            layers[name]=g
    management=list((work/'design').rglob('management.shp'))
    assert len(management)<=1,'Only one management layer allowed'
    if management:
        assert not ('C_change' in layers or 'CN_change' in layers),'Use management.shp OR legacy C/CN layers, not both'
        m=gpd.read_file(management[0])
        assert m.crs is not None and {'parcel_id','c_mult','cn_delta'}<=set(m.columns),'Management layer missing required columns'
        m=m.to_crs(31370)
        ref=gpd.read_file(BASELINE.parent/'reference'/'parcels.gpkg').to_crs(31370)
        ref=ref.loc[ref.LANDUSE!=-4,['parcel_id','geometry']].copy()
        ids=pd.to_numeric(m.parcel_id,errors='raise')
        assert ids.notna().all() and (ids==ids.astype('int64')).all() and ids.is_unique,'Invalid or duplicate parcel IDs'
        m['parcel_id']=ids.astype('int64')
        assert set(m.parcel_id)==set(ref.parcel_id),'Management layer must contain the original eligible parcel IDs'
        m=m.set_index('parcel_id'); ref=ref.set_index('parcel_id').loc[m.index]
        assert m.geometry.is_valid.all() and not m.geometry.is_empty.any()
        assert all(a.symmetric_difference(b).area<=max(0.01,b.area*1e-8) for a,b in zip(m.geometry,ref.geometry)),'Do not alter parcel geometry'
        # Use original geometry and only the two supplied management adjustments.
        m['geometry']=ref.geometry
        for col,lo,hi,neutral,name in [('c_mult',0,1,1,'C_change'),('cn_delta',-20,0,0,'CN_change')]:
            v=pd.to_numeric(m[col],errors='raise')
            assert v.notna().all() and v.between(lo,hi).all(),f'Invalid {col}: expected {lo} to {hi}'
            changed=m.loc[~np.isclose(v,neutral),['geometry']].copy()
            changed[col]=v.loc[changed.index]
            if len(changed): layers[name]=changed.reset_index(drop=True)
        print(f'Edited parcels: {int((~np.isclose(m.c_mult.astype(float),1)|~np.isclose(m.cn_delta.astype(float),0)).sum())}')
    assert layers or management,'No recognized shapefiles in ZIP'
else:
    # Simple reproducible automatic spatial targeting, no ArcGIS required.
    TARGET='Highest erosion' #@param ['All cropland','Random parcels','Highest erosion','Near river']
    TREATED_PERCENT=25 #@param {type:'slider',min:0,max:100,step:5}
    C_MULT=0.6 #@param {type:'number'}
    CN_DELTA=-5 #@param {type:'number'}
    GRASS_WIDTH_M=0 #@param {type:'slider',min:0,max:20,step:1}
    SEED=42 #@param {type:'integer'}
    assert 0<=C_MULT<=1 and -20<=CN_DELTA<=0
    ref=BASELINE.parent/'reference'
    fields=gpd.read_file(ref/'parcels.shp').to_crs(31370)
    rivers=gpd.read_file(ref/'river.shp').to_crs(31370)
    basin=gpd.read_file(ref/'catchment.shp').to_crs(31370)
    fields=gpd.clip(fields,basin); fields=fields[fields.geometry.area>0].copy()
    fields['area']=fields.geometry.area
    if TARGET=='Random parcels': fields=fields.sample(frac=1,random_state=SEED)
    elif TARGET=='Near river': fields=fields.assign(score=fields.geometry.distance(rivers.geometry.union_all())).sort_values('score')
    elif TARGET=='Highest erosion':
        from rasterio.features import geometry_mask
        with rasterio.open(BASELINE/'annual/report/erosion_t_ha.tif') as ds:
            a=ds.read(1); transform=ds.transform
        def score(g):
            m=geometry_mask([g],a.shape,transform,invert=True)&(a!=-9999)
            return float(np.maximum(-a[m],0).mean()) if m.any() else 0.
        fields=fields.assign(score=[score(g) for g in fields.geometry]).sort_values('score',ascending=False)
    if TARGET=='All cropland': selected=fields
    elif TREATED_PERCENT==0: selected=fields.iloc[:0]
    else: selected=fields[fields.area.cumsum().shift(fill_value=0)<fields.area.sum()*TREATED_PERCENT/100]
    # Dissolve selections to eliminate overlaps within each adjustment layer.
    if len(selected):
        geom=selected.geometry.union_all()
        layers['C_change']=gpd.GeoDataFrame({'c_mult':[C_MULT]},geometry=[geom],crs=31370)
        layers['CN_change']=gpd.GeoDataFrame({'cn_delta':[CN_DELTA]},geometry=[geom],crs=31370)
    if GRASS_WIDTH_M>0:
        rivers=gpd.clip(rivers,basin)
        layers['grass']=gpd.GeoDataFrame({'width_m':[GRASS_WIDTH_M]*len(rivers)},geometry=rivers.geometry,crs=31370)
    print('Selected fields:',len(selected),'polygon area ha:',selected.area.sum()/10000,'grass width:',GRASS_WIDTH_M)
    # Export the programmed design as the same shapefile contract.
    design=work/'design'; design.mkdir()
    for name,g in layers.items():
        if len(g): g.to_file(design/f'{name}.shp',index=False)
    (design/'programmed.json').write_text(json.dumps(dict(target=TARGET,treated_percent=TREATED_PERCENT,c_mult=C_MULT,cn_delta=CN_DELTA,grass_width_m=GRASS_WIDTH_M,seed=SEED)))
    shutil.make_archive(str(work/'original'),'zip',design)

cfg=ws.read_ini(BASELINE/'annual/modelinput/inifile.ini'); inp=BASELINE/'annual/modelinput'
prc,prof=ws.read_raster(ws.raster_path(cfg,inp,'Parcel filename')); mask=prc!=0; eligible=prc>0; tr=prof['transform']; res=abs(tr.a)
# Fractional footprint via exact polygon/cell intersection; eligible cropland cells only.
def fraction(geom):
    arr=np.zeros(prc.shape,dtype='float32')
    if geom.is_empty: return arr
    minx,miny,maxx,maxy=geom.bounds
    from rasterio.transform import rowcol
    r0,c0=rowcol(tr,minx,maxy); r1,c1=rowcol(tr,maxx,miny)
    for i in range(max(0,r0),min(arr.shape[0],r1+1)):
        for j in range(max(0,c0),min(arr.shape[1],c1+1)):
            if not eligible[i,j]: continue
            x,y=tr*(j,i); arr[i,j]=geom.intersection(box(x,y-res,x+res,y)).area/res**2
    return arr
def zone_adjust(name,field,lo,hi,neutral):
    a=np.full(prc.shape,neutral,dtype='float32'); used=np.zeros(prc.shape,dtype='float32')
    g=layers.get(name)
    if g is None: return a,used
    assert g.geom_type.isin(['Polygon','MultiPolygon']).all()
    for i in range(len(g)):
        for j in range(i):
            assert g.geometry.iloc[i].intersection(g.geometry.iloc[j]).area<1e-6,'Overlapping zones within same layer'
    for _,r in g.iterrows():
        v=float(r[field]); assert np.isfinite(v) and lo<=v<=hi,(field,v)
        f=fraction(r.geometry); a+=f*(v-neutral); used+=f
    assert used.max()<=1.0001
    return a,used
cm,ca=zone_adjust('C_change','c_mult',0,1,1)
cd,na=zone_adjust('CN_change','cn_delta',-20,0,0)
grass=layers.get('grass'); gf=np.zeros(prc.shape,dtype='float32')
if grass is not None and len(grass):
    assert grass.geom_type.isin(['LineString','MultiLineString']).all()
    widths=pd.to_numeric(grass.width_m); assert widths.between(1,40).all()
    gf=fraction(unary_union([g.buffer(w/2,cap_style=2) for g,w in zip(grass.geometry,widths)]))
areas=dict(C_area_ha=float(ca.sum()*res**2/10000),CN_area_ha=float(na.sum()*res**2/10000),grass_area_ha=float(gf.sum()*res**2/10000))
print(areas)
fig,ax=plt.subplots(1,3,figsize=(14,5))
for a,v,t in zip(ax,[cm,cd,gf],['C multiplier','CN change','Grass fraction']):
    im=a.imshow(np.ma.masked_where(~mask,v)); a.set_title(t); fig.colorbar(im,ax=a,shrink=.6)
plt.show()


## 3. Run and save
Check the design maps, tick ACCEPT, and run. Keep the Scenario ZIP for NB03.

In [ ]:
#@title 3 — Run and download scenario
ACCEPT=False #@param {type:'boolean'}
assert ACCEPT,'Inspect intervention maps and areas first'
# Baseline frozen; each scenario starts from its original native files.
binary=Path('/content/watem_sedem')
urllib.request.urlretrieve('https://github.com/watem-sedem/watem-sedem/releases/download/5.0.3/watem_sedem',binary); binary.chmod(0o755)
DEST.mkdir(parents=True)
shutil.copy2(work/'original.zip',DEST/'design.zip')
(DEST/'design.json').write_text(json.dumps(dict(name=SCENARIO_NAME,**areas,grass_method='fractional C; event-equivalent CN; provisional harmonic kTC with grass kTC=5; PRC/routing unchanged',soil_status='SOC fields descriptive only'),indent=2))
rows=[]
for mode in ['annual','event_40mm']:
    rain=None if mode=='annual' else int(mode.split('_')[1][:-2])
    dest=DEST/mode
    cfg,inp,out=ws.clone_run(BASELINE/mode,dest,binary,rain)
    cp=ws.raster_path(cfg,inp,'C factor map filename'); c,p=ws.read_raster(cp)
    cnew=c.copy(); cnew[eligible]=((c*cm)*(1-gf)+.01*gf)[eligible]
    assert np.isfinite(cnew[mask]).all() and np.all((cnew[mask]>=0)&(cnew[mask]<=1))
    ws.native_write(cp,cnew,p)
    # Illustrative subpixel trapping proxy; not calibrated width-aware routing.
    kp=ws.raster_path(cfg,inp,'ktc map filename'); k,p=ws.read_raster(kp)
    use=eligible & (gf>0)
    k[use]=1/((1-gf[use])/np.maximum(k[use],1e-6)+gf[use]/5.)
    ws.native_write(kp,k,p)
    if rain is not None:
        npth=ws.raster_path(cfg,inp,'CN map filename'); cn,p=ws.read_raster(npth)
        cnnew=cn.copy(); cnnew[eligible]=np.clip(cn[eligible]+cd[eligible],30,100)
        def excess(c):
            S=25400/np.clip(c,1,100)-254; return np.maximum(rain-.2*S,0)**2/np.maximum(rain+.8*S,1e-9)
        # Preserve baseline group B assumption; grass CN=61. Invert event excess.
        target=(1-gf)*excess(cnnew)+gf*excess(np.full(cn.shape,61.))
        low=np.full(cn.shape,1.); high=np.full(cn.shape,100.)
        for i in range(32):
            mid=(low+high)/2; gt=excess(mid)>target; high=np.where(gt,mid,high); low=np.where(gt,low,mid)
        cnnew[use & (target>0)]=((low+high)/2)[use & (target>0)]
        cnnew[use & (target<=0)]=np.minimum(cnnew,61)[use & (target<=0)]
        assert np.all((cnnew[mask]>0)&(cnnew[mask]<=100)); ws.native_write(npth,cnnew,p)
    ws.execute(dest,binary)
    row=ws.report(dest,mode,rain); row.update(areas); rows.append(row)
    ws.tif_write(dest/'report/grass_fraction.tif',gf,prof,mask)
    base,_=ws.read_raster(BASELINE/mode/'report/erosion_t_ha.tif'); cur,_=ws.read_raster(dest/'report/erosion_t_ha.tif')
    ws.tif_write(dest/'report/erosion_difference.tif',cur-base,prof,mask & (cur!=-9999)&(base!=-9999))
    if rain is not None:
        base,_=ws.read_raster(BASELINE/mode/'report/local_runoff_mm.tif'); cur,_=ws.read_raster(dest/'report/local_runoff_mm.tif')
        ws.tif_write(dest/'report/runoff_difference.tif',cur-base,prof,mask & (cur!=-9999)&(base!=-9999))
pd.DataFrame(rows).to_csv(DEST/'summary.csv',index=False); display(pd.DataFrame(rows))
(DEST/'complete.json').write_text(json.dumps(dict(kind='scenario',baseline_id=manifest['baseline_id'],site=SITE_ID,scenario=SCENARIO_NAME,**areas),indent=2))
export=work/'results'; export.mkdir()
for mode in ['annual','event_40mm']: shutil.copytree(DEST/mode/'report',export/mode)
for name in ['design.zip','design.json','summary.csv','complete.json']: shutil.copy2(DEST/name,export/name)
files.download(shutil.make_archive(str(DEST/f'{SCENARIO_NAME}_Scenario'),'zip',export))
